In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.append(str(Path.cwd().parent))

%reload_ext autoreload
%autoreload 2
%matplotlib inline

from src import risk_metrics as rm


In [31]:
print(dir(rm))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'jarque_bera', 'non_normality_check', 'pl']


In [37]:
hfi_pl = pl.read_csv("../data/raw/edhec-hedgefundindices.csv", try_parse_dates=True)
hfi_pl.head()

date,Convertible Arbitrage,CTA Global,Distressed Securities,Emerging Markets,Equity Market Neutral,Event Driven,Fixed Income Arbitrage,Global Macro,Long/Short Equity,Merger Arbitrage,Relative Value,Short Selling,Funds Of Funds
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1997-01-31,1.19,3.93,1.78,7.91,1.89,2.13,1.91,5.73,2.81,1.5,1.8,-1.66,3.17
1997-02-28,1.23,2.98,1.22,5.25,1.01,0.84,1.22,1.75,-0.06,0.34,1.18,4.26,1.06
1997-03-31,0.78,-0.21,-0.12,-1.2,0.16,-0.23,1.09,-1.19,-0.84,0.6,0.1,7.78,-0.77
1997-04-30,0.86,-1.7,0.3,1.19,1.19,-0.05,1.3,1.72,0.84,-0.01,1.22,-1.29,0.09
1997-05-31,1.56,-0.15,2.33,3.15,1.89,3.46,1.18,1.08,3.94,1.97,1.73,-7.37,2.75


In [38]:
rm.non_normality_check(hfi)

Convertible Arbitrage,CTA Global,Distressed Securities,Emerging Markets,Equity Market Neutral,Event Driven,Fixed Income Arbitrage,Global Macro,Long/Short Equity,Merger Arbitrage,Relative Value,Short Selling,Funds Of Funds
bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
true,false,true,true,true,true,true,true,true,true,true,true,true


In [ ]:
# 1. Run your clean check
results_df = rm.non_normality_check(hfi_pl)

# 2. Unpivot to filter for non-normal strategies instantly
non_normal_strategies = (
    results_df.melt()  # Flips wide table layout to a clean long list of column names and values
    .filter(pl.col("value") == True)
    .select("variable")  # Pull out just the strategy names
)

print(non_normal_strategies)

shape: (12, 1)
┌───────────────────────┐
│ variable              │
│ ---                   │
│ str                   │
╞═══════════════════════╡
│ Convertible Arbitrage │
│ Distressed Securities │
│ Emerging Markets      │
│ Equity Market Neutral │
│ Event Driven          │
│ …                     │
│ Long/Short Equity     │
│ Merger Arbitrage      │
│ Relative Value        │
│ Short Selling         │
│ Funds Of Funds        │
└───────────────────────┘


C:\Users\beall\AppData\Local\Temp\ipykernel_36572\3784641985.py:7: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  .melt() # Flips wide table layout to a clean long list of column names and values


In [ ]:
# create a function for non-normality check in polars, using the jarque-bera test from scipy.stats


def non_normality_check_pl(df: pl.DataFrame) -> pl.DataFrame:
    from scipy.stats import jarque_bera

    # Create a new DataFrame to store results
    results = pl.DataFrame()

    for col in df.columns:
        if col != "Date":  # Assuming 'Date' is the only non-numeric column
            stat, p_value = jarque_bera(df[col].drop_nulls().to_numpy())
            results = results.with_columns(
                pl.lit(stat).alias(f"{col}_stat"),
                pl.lit(p_value).alias(f"{col}_p_value"),
                pl.lit(p_value < 0.05).alias(f"{col}_non_normal"),
            )

    return results_df